# 02 — Limpeza e padronização dos dados

Lê a camada **bronze** (`data/parquet/`) e grava a camada **silver** (`data/silver/`), usada pelo record linkage.
A ideia central: aplicar **as mesmas funções de limpeza** na base de busca e no CNEFE, para que os nomes fiquem
comparáveis.

Decisões vindas da EDA (`01_analise_exploratoria.ipynb`):

| Achado | Tratamento |
|---|---|
| `COD_SETOR` tem 16 caracteres terminando em `P` | `setor` = 15 primeiros dígitos |
| Algumas linhas repetem o mesmo `COD_UNICO_ENDERECO` | remover duplicatas |
| CNEFE mistura `2 DE JULHO` e `DOIS DE JULHO` | números por extenso → dígitos |
| Título às vezes separado (`SENHOR` + `DO BONFIM`), às vezes no nome | nome = título + nome |
| Busca tem prefixos sem sentido (`None`, `10R`) e abreviações (`ENG`, `AV`) | remover / expandir |
| Número `SN`, `S/N` ou vazio | número nulo |

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRONZE = ROOT / "data" / "parquet"
SILVER = ROOT / "data" / "silver"

spark = (
    SparkSession.builder.appName("02-limpeza-dados")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "48")
    .config("spark.ui.showConsoleProgress", "false")
    # mantém cod_municipio_6 (pasta da partição) como texto, não número
    .config("spark.sql.sources.partitionColumnTypeInference.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

busca_bronze = spark.read.parquet(str(BRONZE / "base_busca"))
cnefe_bronze = spark.read.parquet(str(BRONZE / "cnefe_bahia"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/11 03:17:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Dicionários

In [2]:
ABREV_TIPO = {
    "R": "RUA", "AV": "AVENIDA", "TV": "TRAVESSA", "TRAV": "TRAVESSA", "ROD": "RODOVIA",
    "EST": "ESTRADA", "FAZ": "FAZENDA", "POV": "POVOADO", "PC": "PRACA", "PCA": "PRACA",
    "AL": "ALAMEDA", "LOT": "LOTEAMENTO", "CJ": "CONJUNTO", "CONJ": "CONJUNTO", "COND": "CONDOMINIO",
    "VL": "VILA", "SIT": "SITIO", "DIST": "DISTRITO", "QD": "QUADRA",
}

ABREV_PALAVRA = {
    "DR": "DOUTOR", "ENG": "ENGENHEIRO", "PROF": "PROFESSOR", "PE": "PADRE", "CEL": "CORONEL",
    "STO": "SANTO", "STA": "SANTA", "SR": "SENHOR", "PRES": "PRESIDENTE", "GOV": "GOVERNADOR",
    "DEP": "DEPUTADO", "VER": "VEREADOR", "GEN": "GENERAL", "MAL": "MARECHAL", "TEN": "TENENTE",
    "JD": "JARDIM", "RESID": "RESIDENCIAL",
}

NUMEROS_EXTENSO = {
    "UM": "1", "DOIS": "2", "TRES": "3", "QUATRO": "4", "CINCO": "5", "SEIS": "6", "SETE": "7",
    "OITO": "8", "NOVE": "9", "DEZ": "10", "ONZE": "11", "DOZE": "12", "TREZE": "13",
    "QUATORZE": "14", "QUINZE": "15", "DEZESSEIS": "16", "DEZESSETE": "17", "DEZOITO": "18",
    "DEZENOVE": "19", "VINTE": "20", "TRINTA": "30",
}

STOPWORDS = ["DE", "DA", "DO", "DAS", "DOS", "E", "D"]
ZONA_RURAL = ["ZONA RURAL", "RURAL"]

## 2. Funções de limpeza

In [3]:
def normalizar_texto(c):
    """Maiúsculas, sem acentos e pontuação, sem 'None', espaços simples. Texto vazio vira nulo."""
    c = F.translate(F.upper(c), "ÁÀÂÃÉÊÍÓÔÕÚÇ", "AAAAEEIOOOUC")
    c = F.regexp_replace(c, r"[^A-Z0-9 ]", " ")
    c = F.regexp_replace(c, r"\b(NONE|NULL)\b", " ")
    c = F.trim(F.regexp_replace(c, r"\s+", " "))
    return F.when(c != "", c)


def trocar_palavras(c, dicionario):
    """Troca cada palavra do texto pela entrada do dicionário (se existir)."""
    mapa = F.create_map([F.lit(x) for par in dicionario.items() for x in par])
    palavras = F.transform(F.split(c, " "), lambda p: F.coalesce(mapa[p], p))
    return F.array_join(palavras, " ")


def normalizar_nome(c):
    """Expande abreviações e converte números por extenso (VINTE E UM → 21)."""
    c = trocar_palavras(c, {**ABREV_PALAVRA, **NUMEROS_EXTENSO})
    return F.regexp_replace(c, r"\b([23])0 (E )?([1-9])\b", "$1$3")


def chave_nome(c):
    """Nome sem DE/DA/DO. Se só sobrar stopword (RUA E), mantém o nome."""
    sem_stopwords = F.trim(F.regexp_replace(
        F.regexp_replace(c, r"\b(" + "|".join(STOPWORDS) + r")\b", " "), r"\s+", " "))
    return F.when(sem_stopwords != "", sem_stopwords).otherwise(c)


def tipo_base(c):
    """1ª palavra do tipo, ignorando ordinais: '1A TRAVESSA DA RUA' → TRAVESSA."""
    return F.split(F.regexp_replace(c, r"^\d+A ", ""), " ")[0]

In [4]:
# Teste rápido com casos reais da base de busca
casos = ["None EVILASIO FIRMATO", "RUA VINTE UM DE ABRIL", "RUA SENHOR DO BOMFIM", "DISTRITO ENG PONTES",
         "Avenida São João", "RUA E"]
teste = spark.createDataFrame([(c,) for c in casos], "entrada string")
teste = teste.withColumn("nome", normalizar_nome(normalizar_texto(F.col("entrada"))))
teste.withColumn("chave", chave_nome(F.col("nome"))).toPandas()

,entrada,nome,chave
0,None EVILASIO FIRMATO,EVILASIO FIRMATO,EVILASIO FIRMATO
1,RUA VINTE UM DE ABRIL,RUA 21 DE ABRIL,RUA 21 ABRIL
2,RUA SENHOR DO BOMFIM,RUA SENHOR DO BOMFIM,RUA SENHOR BOMFIM
3,DISTRITO ENG PONTES,DISTRITO ENGENHEIRO PONTES,DISTRITO ENGENHEIRO PONTES
4,Avenida São João,AVENIDA SAO JOAO,AVENIDA SAO JOAO
5,RUA E,RUA E,RUA


## 3. Tipos de logradouro

Em vez de digitar uma lista, uso os tipos que o próprio CNEFE usa (com pelo menos 100 endereços).

In [5]:
tipos = (
    cnefe_bronze.select(tipo_base(normalizar_texto(F.col("NOM_TIPO_SEGLOGR"))).alias("tipo"))
    .groupBy("tipo").count().where("count >= 100")
)
TIPOS = sorted({r["tipo"] for r in tipos.collect()} | set(ABREV_TIPO.values()))
print(len(TIPOS), "tipos:", ", ".join(TIPOS))

164 tipos: ACAMPAMENTO, ACESSO, ACUDE, AEROPORTO, AGLOMERADO, AGROVILA, ALAMEDA, ALDEIA, ALTO, ANEL, AREA, ARRAIAL, ASSENTAMENTO, ATALHO, AVENIDA, BAIA, BAIRRO, BAIXA, BAIXADA, BAIXAO, BARRA, BARRAGEM, BECO, BEIRA, BEIRADA, BLOCO, BOQUEIRAO, BREJO, CABECEIRA, CACHOEIRA, CALCADA, CAMINHO, CAMPO, CANAL, CAPAO, CENTRO, CERCA, CHACARA, CHAPADA, COHAB, COLONIA, COMUNIDADE, CONDOMINIO, CONJUNTO, CONTINUACAO, CONTORNO, CORREDOR, CORREGO, CRUZAMENTO, DESCIDA, DESVIO, DISTRITO, DIVISA, EDIFICIO, EIXO, ENGENHO, ENTRADA, ENTRONCAMENTO, ESCADA, ESCADARIA, ESTACAO, ESTANCIA, ESTRADA, EXTENSAO, FAVELA, FAZENDA, FINAL, FLORESTA, FUNDOS, GAMELEIRA, GARIMPO, GROTA, HABITACIONAL, HARAS, ILHA, INVASAO, JARDIM, LADEIRA, LADO, LAGO, LAGOA, LARGO, LATERAL, LIGACAO, LIMITE, LINHA, LOTE, LOTEAMENTO, LUGAREJO, MARGEM, MARGENS, MATA, MATO, MONTE, MORRO, NUCLEO, ORLA, OUTROS, PARALELA, PARQUE, PASSAGEM, PASSARELA, PASSEIO, PEDRA, PERIMETRO, PONTA, PONTE, PONTO, PORTO, POSTO, POVOADO, PRACA, PRAIA, PROJETADA, PRO

## 4. Base de busca → silver

O logradouro vem num campo só (`RUA SERGIO BOMFIM`), então separo **tipo** (1ª palavra) e **nome** (resto).
Se a 1ª palavra não é tipo mas a 2ª é (`10R FAZENDA BOA ESPERANCA`), descarto a 1ª.

In [6]:
def tipo_de(palavra):
    mapa = F.create_map([F.lit(x) for par in ABREV_TIPO.items() for x in par])
    return F.coalesce(mapa[palavra], palavra)


palavras = F.split(normalizar_texto(F.col("consulta_logradouro")), " ")
lixo_no_inicio = ~tipo_de(palavras[0]).isin(TIPOS) & tipo_de(palavras[1]).isin(TIPOS)
palavras = F.when(lixo_no_inicio, F.slice(palavras, 2, 100)).otherwise(palavras)
tem_tipo = tipo_de(palavras[0]).isin(TIPOS)

busca = (
    busca_bronze
    .withColumn("cod_municipio_6", F.col("consulta_municipio"))
    .withColumn("tipo_norm", F.when(tem_tipo, tipo_de(palavras[0])))
    .withColumn("nome_norm", normalizar_nome(F.array_join(
        F.when(tem_tipo, F.slice(palavras, 2, 100)).otherwise(palavras), " ")))
    .withColumn("nome_chave", chave_nome(F.col("nome_norm")))
    .withColumn("numero", F.regexp_extract("consulta_numero", r"^(\d+)", 1).cast("int"))
    .withColumn("numero", F.when(F.col("numero") > 0, F.col("numero")))
    .withColumn("cep", F.col("consulta_cep"))
    .withColumn("cep_generico", F.col("cep").endswith("000"))
    .withColumn("bairro_norm", normalizar_nome(normalizar_texto(F.col("consulta_bairro"))))
    .withColumn("bairro_norm", F.when(~F.col("bairro_norm").isin(ZONA_RURAL), F.col("bairro_norm")))
    .orderBy("source_row_number")
)

bp = busca.toPandas()
assert len(bp) == 107, "a limpeza não pode perder nem criar registros"
bp[["consulta_logradouro", "tipo_norm", "nome_norm", "nome_chave", "consulta_numero", "numero", "bairro_norm"]]

,consulta_logradouro,tipo_norm,nome_norm,nome_chave,consulta_numero,numero,bairro_norm
0,RODOVIA BR 101,RODOVIA,BR 101,BR 101,None,NaN,BR101
1,RUA RIO BAIANO,RUA,RIO BAIANO,RIO BAIANO,SN,NaN,CENTRO
2,FAZENDA BICO,FAZENDA,BICO,BICO,None,NaN,None
3,DISTRITO 2 DE JULHO,DISTRITO,2 DE JULHO,2 JULHO,22,22.0,POCO CENTRAL
4,RUA SERGIO BOMFIM,RUA,SERGIO BOMFIM,SERGIO BOMFIM,332,332.0,BARREIRINHAS
...,...,...,...,...,...,...,...
102,RUA SEM DENOMINACAO,RUA,SEM DENOMINACAO,SEM DENOMINACAO,26,26.0,POVOADO DE JACAREZINHO
103,RUA NOVA,RUA,NOVA,NOVA,78,78.0,NAZARE DE JACUIPE
104,EIXO ESTRADA VICINAL 01,EIXO,ESTRADA VICINAL 01,ESTRADA VICINAL 01,15,15.0,AGROVILA 07
105,RUA D,RUA,D,D,169,169.0,None


## 5. CNEFE → silver

In [7]:
# Antes de remover duplicatas: um mesmo endereço nunca está em dois setores?
duplicados_em_setores = (
    cnefe_bronze.groupBy("COD_UNICO_ENDERECO").agg(F.countDistinct("COD_SETOR").alias("n"))
    .where("n > 1").count()
)
assert duplicados_em_setores == 0

cnefe = (
    cnefe_bronze
    .dropDuplicates(["COD_UNICO_ENDERECO"])
    .select(
        "COD_UNICO_ENDERECO", "cod_municipio_6", "COD_SETOR",
        F.substring("COD_SETOR", 1, 15).alias("setor"),
        F.col("CEP").alias("cep"),
        tipo_base(normalizar_texto(F.col("NOM_TIPO_SEGLOGR"))).alias("tipo_norm"),
        normalizar_nome(normalizar_texto(F.concat_ws(" ", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR"))).alias("nome_norm"),
        normalizar_nome(normalizar_texto(F.col("DSC_LOCALIDADE"))).alias("localidade_norm"),
        F.when(F.col("NUM_ENDERECO").cast("int") > 0, F.col("NUM_ENDERECO").cast("int")).alias("numero"),
    )
    .withColumn("nome_chave", chave_nome(F.col("nome_norm")))
)
cnefe.limit(5).toPandas()

,COD_UNICO_ENDERECO,cod_municipio_6,COD_SETOR,setor,cep,tipo_norm,nome_norm,localidade_norm,numero,nome_chave
0,100009250,292610,292610305000008P,292610305000008,48750000,POVOADO,ALGODOES,ALGODOES,38.0,ALGODOES
1,100009264,292610,292610305000007P,292610305000007,48750000,FAZENDA,RIACHO FUNDO,RIACHO FUNDO,NaN,RIACHO FUNDO
2,100009269,290320,290320105000296P,290320105000296,47800028,SITIO,BARRIGUDA,SITIO DA BARRIGUDA,NaN,BARRIGUDA
3,100009280,290320,290320105000297P,290320105000297,47800028,POVOADO,ARRAIAL DA PENHA,ARRAIAL DA PENHA,NaN,ARRAIAL PENHA
4,100009291,290320,290320105000296P,290320105000296,47800028,RODOVIA,BR 242,RODOVIA BR 242,NaN,BR 242


## 6. Gravação

In [8]:
busca.coalesce(1).write.mode("overwrite").parquet(str(SILVER / "busca"))
cnefe.write.mode("overwrite").partitionBy("cod_municipio_6").parquet(str(SILVER / "cnefe"))

busca_silver = spark.read.parquet(str(SILVER / "busca"))
cnefe_silver = spark.read.parquet(str(SILVER / "cnefe"))
print(f"busca: {busca_silver.count()} linhas | cnefe: {cnefe_silver.count():,} linhas")

busca: 107 linhas | cnefe: 8,528,506 linhas


## 7. A limpeza ajudou?

Quantos endereços da busca têm um nome **idêntico** no CNEFE do mesmo município. Na EDA, sem limpeza, eram 42 de 107.

In [9]:
nomes_cnefe = cnefe_silver.select("cod_municipio_6", "nome_chave").distinct()
chave = ["cod_municipio_6", "nome_chave"]
print("com nome idêntico após a limpeza:", busca_silver.join(nomes_cnefe, chave, "left_semi").count(), "de 107")

# Os que ainda não batem vão depender da comparação aproximada (fuzzy) no linkage
busca_silver.join(nomes_cnefe, chave, "left_anti").select("consulta_logradouro", "nome_chave").toPandas()

com nome idêntico após a limpeza: 64 de 107


,consulta_logradouro,nome_chave
0,FAZENDA PERSEVERANCA,PERSEVERANCA
1,RUA NOVA BRASILEIA,NOVA BRASILEIA
2,RUA ADELINO SUZART PEREIRA,ADELINO SUZART PEREIRA
3,RUA MANOEL ALVES SAMPAIO,MANOEL ALVES SAMPAIO
4,FAZENDA PALMEIRAS DO RIACHO,PALMEIRAS RIACHO
5,RUA ALFREDO VEIGA,ALFREDO VEIGA
6,DISTRITO TAGUA,TAGUA
7,RUA ADEGUNDES ARAUJO,ADEGUNDES ARAUJO
8,RUA SERGIO BOMFIM,SERGIO BOMFIM
9,RUA CAMINHO 20 CASA,CAMINHO 20 CASA


In [10]:
spark.stop()